# Fine-tune or not? The BANKING77 sweep (Kaggle GPU)

Runs the whole learning-curve sweep from `EVAL_PLAN.md`: three arms (TF-IDF + LR, frozen
`bge-small` + LR, fine-tuned DistilBERT) at k = 5, 10, 25, 50 and all labelled examples per
intent, three seeds each, then builds the report from the saved results.

**Notebook settings before *Run all*:**

1. *Accelerator*: **GPU T4**. The fine-tuning arm needs it.
2. *Internet*: **on** (pip, GitHub, the dataset CSVs, model weights).
3. Only if you set `PUSH_TO_HUB = True`: *Add-ons, Secrets*: **`HF_TOKEN`**, attached to this notebook.

The runtime is not known until the first run; the sweep is resumable, so if the session drops,
re-run the notebook and it continues from `results/runs.csv`.

Pin `REF` to a commit SHA before the run that counts: all 45 runs must come from one SHA.

In [ ]:
# --- Run configuration: everything that defines this run is here -----------------------------
REPO = "https://github.com/Prithv122/finetune-transformer"
REF = "main"  # pin a commit SHA before the run that counts

ARMS = "tfidf,embed,finetune"
KS = "5,10,25,50,all"
SEEDS = "1,2,3"

# To continue a sweep from an earlier session: attach that session's output as an input and
# point this at its results folder, e.g. "/kaggle/input/<notebook-output>/results".
RESUME_FROM = None

PUSH_TO_HUB = False
HUB_REPO = "Prithv122/distilbert-banking77"

## 1. Clone at the pinned ref

In [ ]:
import subprocess
from pathlib import Path

WORK = Path("/kaggle/working/finetune-transformer")
if not WORK.exists():
    subprocess.run(["git", "clone", "-q", REPO, str(WORK)], check=True)
subprocess.run(["git", "-C", str(WORK), "fetch", "-q", "origin"], check=True)
# A branch name resolves to the fetched remote branch, not a stale local one; a SHA as itself.
remote = subprocess.run(["git", "-C", str(WORK), "rev-parse", "--verify", "-q", f"origin/{REF}"],
                        capture_output=True, text=True)
target = f"origin/{REF}" if remote.returncode == 0 else REF
subprocess.run(["git", "-C", str(WORK), "checkout", "-q", "--detach", target], check=True)
COMMIT = subprocess.run(
    ["git", "-C", str(WORK), "rev-parse", "HEAD"], capture_output=True, text=True, check=True
).stdout.strip()
print("repo at", COMMIT)

## 2. Dependencies, keeping Kaggle's CUDA torch

The package is not pip-installed: `pyproject.toml` pins Python 3.12 and Kaggle's image may run
another version, so the code runs from `src/` on `PYTHONPATH` (it is kept 3.11-compatible).
Only dependencies that are missing get installed, and none of them may pull in a different torch.

In [ ]:
import importlib.util
import os
import shutil
import sys

os.chdir(WORK)
os.environ["PYTHONPATH"] = str(WORK / "src")  # inherited by the !python cells below
sys.path.insert(0, str(WORK / "src"))
print("python", sys.version.split()[0])

missing = [
    pip_name
    for module, pip_name in [
        ("numpy", "numpy>=2.0"),
        ("sklearn", "scikit-learn>=1.5"),
        ("matplotlib", "matplotlib>=3.9"),
        ("transformers", "transformers>=4.45"),
        ("sentence_transformers", "sentence-transformers>=3.0"),
    ]
    if importlib.util.find_spec(module) is None
]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

import sentence_transformers
import torch
import transformers

print("torch", torch.__version__)
print("transformers", transformers.__version__)
print("sentence_transformers", sentence_transformers.__version__)
assert torch.cuda.is_available(), "no GPU: set the accelerator to GPU T4 and restart"
print("gpu", torch.cuda.get_device_name(0))

## 3. Download the pinned BANKING77 CSVs

In [ ]:
!{sys.executable} -m finetune.cli data-download --dest data/raw/banking77

## 4. The sweep

Resumable: a run whose row and predictions file are both in `results/` is skipped. Within a
session, re-run this cell after an error; from a new session, set `RESUME_FROM` first
(`/kaggle/working` does not survive a new session). Resume only with the same `REF`: the report flags a
`runs.csv` that mixes git SHAs, because the plan asks for one sweep at one commit.

In [ ]:
if RESUME_FROM:
    # Copy the earlier session's runs.csv and preds/ in; the sweep skips every run already there.
    shutil.copytree(RESUME_FROM, WORK / "results", dirs_exist_ok=True)
    print("resumed from", RESUME_FROM)

!{sys.executable} -m finetune.cli sweep --data-dir data/raw/banking77 --out results --arms {ARMS} --ks {KS} --seeds {SEEDS} --keep-checkpoint all:1

## 5. Report

In [ ]:
!{sys.executable} -m finetune.cli report --results results

from IPython.display import Markdown, display

display(Markdown(Path("results/summary.md").read_text(encoding="utf-8")))

## 6. Zip the results for download

In [ ]:
import shutil

archive = shutil.make_archive("/kaggle/working/results", "zip", root_dir=WORK, base_dir="results")
print(archive, round(Path(archive).stat().st_size / 1e6, 1), "MB")

## 7. Optional: push the k = all, seed 1 model to the Hub

Only runs when `PUSH_TO_HUB = True`. Reads `HF_TOKEN` from Kaggle secrets; the token is never
printed.

In [ ]:
if PUSH_TO_HUB:
    from huggingface_hub import HfApi
    from kaggle_secrets import UserSecretsClient

    api = HfApi(token=UserSecretsClient().get_secret("HF_TOKEN"))
    model_dir = WORK / "models" / "finetune_kall_s1"
    # The model card goes up with the weights; it quotes numbers from results/ only.
    shutil.copy(WORK / "MODEL_CARD.md", model_dir / "README.md")
    api.create_repo(HUB_REPO, repo_type="model", exist_ok=True)
    api.upload_folder(
        folder_path=str(model_dir),
        repo_id=HUB_REPO,
        repo_type="model",
        commit_message=f"DistilBERT fine-tuned on BANKING77, all labels, seed 1 (repo {COMMIT[:7]})",
    )
    print("pushed to", HUB_REPO)
else:
    print("PUSH_TO_HUB is False: nothing pushed")